# 0. Project: Image Captioninog on FLickr dataset

In [ ]:
pip install transformers

In [ ]:
pip install git+https://github.com/openai/CLIP.git

# 1. Imports

In [ ]:
# === Standard Library ===
import os
import gc
import random
import string
from glob import glob
from collections import Counter, defaultdict

# === Data Science & Visualization ===
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

# === Natural Language Processing ===
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

# === PyTorch Core ===
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.nn.utils.rnn import pad_sequence
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset, DataLoader

# === Torchvision (Computer Vision) ===
import torchvision
from torchvision import transforms, models

# === Hugging Face Transformers ===
from transformers import (
    GPT2Tokenizer,
    GPT2LMHeadModel,
    BertTokenizer,
    BlipProcessor,
    BlipForConditionalGeneration,
    get_linear_schedule_with_warmup,
    Blip2Processor,
    Blip2ForConditionalGeneration
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# 2. Extract Data

In [ ]:
# Paths
text_extract_path = '/content/drive/MyDrive/Deep Learning/Flickr8k_text'
dataset_extract_path = '/content/drive/MyDrive/Deep Learning/Flickr8k_Dataset/Flicker8k_Dataset'

In [ ]:
# Load lammentized
caption_file_name = 'Flickr8k.token.txt'

In [ ]:
# Full file paths
captions_path      = os.path.join(text_extract_path, caption_file_name)
train_images_path  = os.path.join(text_extract_path, 'Flickr_8k.trainImages.txt')
val_images_path    = os.path.join(text_extract_path, 'Flickr_8k.devImages.txt')
test_images_path   = os.path.join(text_extract_path, 'Flickr_8k.testImages.txt')


In [ ]:
# Load the captions file
captions_df = pd.read_csv(
    captions_path,
    sep='\t',
    names=['image_caption', 'caption']
)

# Extract image filename without the # index
captions_df['image'] = captions_df['image_caption'].apply(lambda x: x.split('#')[0])

In [ ]:
# Load splits
def load_image_list(filepath):
    with open(filepath, 'r') as f:
        images = f.read().strip().split('\n')
    return set(images)

In [ ]:
train_images = load_image_list(train_images_path)
val_images = load_image_list(val_images_path)
test_images = load_image_list(test_images_path)

print(f"Train Images: {len(train_images)}, val Images: {len(val_images)}, Test Images: {len(test_images)}")


In [ ]:
# Filter dataframe based on splits
train_captions_df = captions_df[captions_df['image'].isin(train_images)]
val_captions_df = captions_df[captions_df['image'].isin(val_images)]
test_captions_df = captions_df[captions_df['image'].isin(test_images)]

print(f"Train Captions: {len(train_captions_df)}, Dev Captions: {len(val_captions_df)}, Test Captions: {len(test_captions_df)}")


In [ ]:
import string

def clean_caption(caption):
    caption = caption.lower()
    caption = caption.translate(str.maketrans('', '', string.punctuation))
    caption = caption.strip()
    return caption

train_captions_df['caption'] = train_captions_df['caption'].apply(clean_caption)
val_captions_df['caption'] = val_captions_df['caption'].apply(clean_caption)
test_captions_df['caption'] = test_captions_df['caption'].apply(clean_caption)



## [2.1] Plot a Sample Image

In [ ]:
img_name = random.choice(list(val_images))
img_path = os.path.join(dataset_extract_path, img_name)
captions = val_captions_df[val_captions_df['image'] == img_name]['caption'].values

img = Image.open(img_path)
plt.imshow(img)
plt.axis('off')
plt.title('\n'.join(captions), fontsize=10)
plt.show()

# Create Encoder-Decoder Network

### Build Vocab

In [ ]:
# 1. Initialize the tokenizer (use BERT from huggingface)
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [ ]:
# 2. Inspect its properties
vocab_size = tokenizer.vocab_size         # typically 30522
pad_idx    = tokenizer.pad_token_id       # usually 0
start_idx  = tokenizer.cls_token_id       # [CLS] token id, usually 101
end_idx    = tokenizer.sep_token_id       # [SEP] token id, usually 102

print(f"Vocab size: {vocab_size}")
print(f"[PAD] token id: {pad_idx}")
print(f"[CLS] token id: {start_idx}")
print(f"[SEP] token id: {end_idx}")

## 3. Experiment 1: Create Baseline model (MobileNetV3-small encoder + LSTM Decoder & BERT Tokenizer)

+ Intially, MobileNetV2 is chosen because it is lightweight with al ow parameter count and fast inference, compared to other ImageNet pretrained model. As a baseline we do not need the best possible vision backbone. We just need something that works reasonably well to validate the full pipeline (data -> model -> caption -> evaluation

+ BERT Tokenizer uses WordPiece tokens, which are highly expressive and help manage out-of-vocabulary words better than simple word-level tokenizers

### [3.1] Dataset & Dataloader

1. **FlickrDataset Class**

Defines a custom PyTorch `Dataset` that loads image-caption pairs, applies image transformations, and tokenizes captions for use in deep learning models.

2. **collate_fn Function**

Handles batching of variable-length tokenized captions by padding them to equal length and stacking image tensors for efficient training.

3. **DataLoader Setup**

Wraps the dataset to enable mini-batch loading, optional shuffling, and custom batching logic via `collate_fn`, making training and validation efficient.

4.** pad_idx**

Specifies the token ID used for padding captions, ensuring consistent sequence length during batching, which is crucial for training sequence models.

Essentially, this class prepare the combination of Image and Captions for the DataLoader. It return a tuple of batched images, padded caption and image name (need the name to look up correct 5 references for BLEU evaluation)

In [ ]:
class FlickrDataset(Dataset):
    def __init__(self, df, img_dir, tokenizer, transform, max_len=50):
        self.df        = df.reset_index(drop=True)  # Dataframe innit
        self.img_dir   = img_dir                    # Directory where images are stored
        self.tokenizer = tokenizer                  # Tokenizer
        self.transform = transform                  # Image transformation defined
        self.max_len   = max_len                    # Max caption token

    def __len__(self):
        return len(self.df)       # Return total numbers of (image, caption) pairs

    def __getitem__(self, idx):   # Load one sample at a time
        row = self.df.iloc[idx]

        # 1) Load & transform the image
        img = Image.open(os.path.join(self.img_dir, row['image'])).convert('RGB')  # Load image from disk & convert to RGB
        img = self.transform(img)  # Apply tranformation

        # 2) Use Tokenizer to encode the caption to a LIST of token IDs
        token_ids = self.tokenizer.encode(
            row['caption'],            # caption is a pure Python str
            add_special_tokens=True,   # adds [CLS] & [SEP]
            max_length=self.max_len,
            truncation=True            # Truncates if its longer than max_len
        )
        caption_ids = torch.tensor(token_ids, dtype=torch.long)  # Convert tokens to tensor

        # 3) A tuple of the transformed image, encoded caption, and the image filename
        return img, caption_ids, row['image']

# Define how to collate (combine) individual sample into batches
def collate_fn(batch):
    images, captions, names = zip(*batch)           # Seperate the component of each sample
    images   = torch.stack(images, dim=0)           # Combine all iamge tensors into a batch
    captions = pad_sequence(captions,               # Pad all caption sequences to the length of the longest one in the batch with pad_idx
                            batch_first=True,
                            padding_value=pad_idx)
    return images, captions, names                  # Return a tuple of bathced images, padded captions and image name (need name to look up correct 5 references for BLEU evaluation)


This set up allow:

*   Handles variable-length captions with padding.
*   Separates image preprocessing from text tokenization cleanly.
*   Easily plug into any torch.nn model using a DataLoader





### [3.2] EncoderCNN (MobileNet_V3_Small)

In [ ]:
class EncoderCNN(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        backbone = models.mobilenet_v3_small(weights=models.MobileNet_V3_Small_Weights.IMAGENET1K_V1) # Loads the pre-trained MobileNetV3-Small model with ImageNet weights
        modules  = list(backbone.children())[:-1]       # Strips off the classifier layer (usually the last one), leaving only the feature extraction layers
        self.cnn  = nn.Sequential(*modules)             # Wraps them in a nn.Sequential

        with torch.no_grad():                               # Sends a dummy image through self.cnn to figure out the feature dimension size (feat_dim) of the CNN output after flattening.
            sample   = torch.zeros(1,3,224,224)             # Using a dummy input helps compute this without manually checking MobileNet's architecture
            feat_dim = self.cnn(sample).view(1,-1).size(1)

        self.proj = nn.Linear(feat_dim, embed_size)             # Adds a linear projection layer to map high-dimensional CNN features into a smaller embedding space (embed_size). This is done for use in later models like RNNs or attention modules
        self.bn   = nn.BatchNorm1d(embed_size, momentum=0.01)   # Adds batch normalization to stabilize and normalize the embeddings.
        for p in self.cnn.parameters():                         # Freezes the CNN’s weights — only the projection and batch norm layers will be trained
            p.requires_grad = False

    def forward(self, x):
        feats = self.cnn(x)                         # Passes input images x (shape [B, 3, 224, 224]) through the CNN backbone to extract features.
        B,C,H,W = feats.size()                      # Reshapes from [B, C, H, W] → [B, C, H*W]
        feats = feats.view(B,C,-1).permute(0,2,1)   # permutes to [B, H*W, C]. This makes each spatial location (e.g., 7×7=49 tokens) into a sequence of vectors — ideal for feeding into sequential models like LSTMs or Transformers that operate on sequences.
        proj  = self.proj(feats)                    # Applies the projection layer to each spatial feature vector: [B, H*W, C] → [B, H*W, embed_size]
        flat  = proj.reshape(-1, proj.size(-1))     # Applies batch normalization across the flattened [B*H*W, embed_size] dimension
        bn    = self.bn(flat).view_as(proj)         # reshapes it back to [B, H*W, embed_size]
        return bn


+ Final output: a batch of image features of shape [B, H*W, embed_size].

+ Each image is represented as a sequence of H*W feature vectors of dimension embed_size — ideal for feeding into an attention-based decoder like an LSTM or Transformer.



### [3.3] Attention

The goal of attentionis to compute weight (importance) for each saptial image feature (from the encoder) based on the current decoder hidden state (h).

In [ ]:
class Attention(nn.Module):
    def __init__(self, feat_dim, hid_dim):
        super().__init__()
        self.attn = nn.Linear(feat_dim + hid_dim, hid_dim)      # Takes a concatenation of image feature vectors (dim = feat_dim) and decoder hidden state (dim = hid_dim). Projects it into a shared attention space of dimension hid_dim.
        self.v    = nn.Linear(hid_dim, 1)                       # Map the hidden attention scores to a single scaler score per feature vector

    def forward(self, feats, h):
        if h.dim() == 3:      # Handles case where h comes as a tuple (e.g., from LSTM: (h_n, c_n)).
            h = h[0]          # Picks only the hidden state part, not the cell state
        h_exp   = h.unsqueeze(1).repeat(1, feats.size(1), 1)  # Expands decoder hidden state h from shape [B, hid_dim] to [B, T, hid_dim], where T = number of image features (e.g., 49)
        energy  = torch.tanh(self.attn(torch.cat([feats, h_exp], dim=2)))    # Concatenates each image feature with the decoder state: [B, T, feat_dim + hid_dim]. Passes through a linear layer and tanh nonlinearity to compute energy scores — how well each feature aligns with the decoder state.
        weights = self.v(energy).squeeze(2)   # Passes energy scores through another linear layer to get unnormalized attention scores [B, T, 1]. Squeezes to shape [B, T], one score per feature per image in the batch.
        return torch.softmax(weights, dim=1)  # Applies softmax over the time/sequence dimension to get attention weights


### [3.4] DecoderLSTM

In [ ]:
class DecoderLSTM(nn.Module):
    def __init__(self, embed_size, hidden_size, vocab_size, feat_dim):
        super().__init__()
        self.embed     = nn.Embedding(vocab_size, embed_size)                           #  Converts token indices into dense word embeddings for input to LSTM.
        self.lstm      = nn.LSTM(embed_size + feat_dim, hidden_size, batch_first=True)  #  LSTM takes word embedding + attention context vector at each time step.
        self.linear    = nn.Linear(hidden_size, vocab_size)                             #  Projects LSTM outputs into logits over vocabulary for word prediction.
        self.init_h    = nn.Linear(feat_dim, hidden_size)
        self.init_c    = nn.Linear(feat_dim, hidden_size)                               #  Initializes LSTM hidden (h) and cell (c) states from the mean-pooled CNN features.
        self.attention = Attention(feat_dim, hidden_size)                               #  Computes attention weights over spatial features (see earlier explanation).

    def forward(self, features, captions):            # Tensor of shape (B,N,D) where B = batch size, N = number of spatial regions, D = feature dimension per region
        embeddings = self.embed(captions[:, :-1])     # Applies an embedding layer to map roken incies to dense vectors (B, T, E)
        B, T, _    = embeddings.size()

        h = self.init_h(features.mean(dim=1)).unsqueeze(0)  # Compute average of image feature accross spatial region (1, B, D)
        c = self.init_c(features.mean(dim=1)).unsqueeze(0)  # Pass through linear layers (init_h, init_c) to generate initial hidden (h) and cell (c) states for LTSM. Note: The .unsqueeze(0) adds the LSTM’s expected time dimension: (num_layers, B, H), typically (1, B, H)

        outputs = []
        for t in range(T):          # Iterative Decoding for each timestep
            h_s   = h.squeeze(0)                          # Current hidden state
            a     = self.attention(features, h_s)         # Attention weights over image region
            ctx   = torch.bmm(a.unsqueeze(1), features)   # Context vector weighted sum over image features
            e_t   = embeddings[:, t:t+1, :]               # Embedding of current word token
            inp   = torch.cat([e_t, ctx], dim=2)          # Concatenate (embedding + context) to feed into LTSM
            out, (h, c) = self.lstm(inp, (h, c))          # LSTM output at timestep t, appended to outputs.
            outputs.append(out)

        outputs = torch.cat(outputs, dim=1)       # Concatenates LSTM outputs over all timesteps into a sequence.
        return self.linear(outputs)               # Passes it through a linear layer to map hidden states to vocabulary logits (for softmax) → V is the vocabulary size


# This function implements beam search decoding for a sequence generation model

    def beam_search(self, features, max_len=20, start_token_id=101, end_token_id=102, k=3):
        device = features.device
        # features: [1, N, feat_dim]
        feat_mean = features.mean(dim=1)               # [1, feat_dim], Initialize the hidden and cells state of the LSTM using the mean-pooled features
        h = self.init_h(feat_mean).unsqueeze(0)        # [1, B=1, hidden]
        c = self.init_c(feat_mean).unsqueeze(0)

        # Initialize beam: (sequence, score, h, c); sequence of token IDs, cumulative log-probability score, hidden & cell state
        beams = [([start_token_id], 0.0, h, c)]

        for _ in range(max_len):
            all_candidates = []
            for seq, score, h, c in beams:
                if seq[-1] == end_token_id:
                    all_candidates.append((seq, score, h, c))
                    continue

                # 1) embed last token
                last_token = torch.tensor([seq[-1]], device=device)
                inp = self.embed(last_token).unsqueeze(1)  # [1,1,embed]

                # 2) attention over the full feature sequence
                attn_w = self.attention(features, h.squeeze(0))      # [1, N]
                ctx    = torch.bmm(attn_w.unsqueeze(1), features)    # [1,1,feat_dim]

                # 3) LSTM step
                lstm_in = torch.cat([inp, ctx], dim=2)               # [1,1,embed+feat]
                out, (h_new, c_new) = self.lstm(lstm_in, (h, c))     # out: [1,1,hidden]

                # 4) project to vocab and get top-k
                logits    = self.linear(out.squeeze(1))              # [1, vocab_size]
                log_probs = torch.log_softmax(logits, dim=1)         # [1, vocab_size]
                topk_lp, topk_idx = log_probs.topk(k, dim=1)         # each is [1, k]

                # 5) build new beams
                for j in range(k):
                    new_seq   = seq + [topk_idx[0, j].item()]
                    new_score = score + topk_lp[0, j].item()
                    all_candidates.append((new_seq, new_score, h_new, c_new))

            # prune to top-k: We keep only k hiest scoring sequences
            beams = sorted(all_candidates, key=lambda x: x[1], reverse=True)[:k]
            if all(b[0][-1] == end_token_id for b in beams):
                break

        best_seq = beams[0][0]
        return best_seq[1:] # Return the best-scoring sequence, exclusing the start token

This is a soft attention decoder that:

1. Receives image features from a CNN encoder.

2. Uses attention at each decoding step to focus on relevant image regions.

3. Generates a word (logits) at each step based on:

*   The previous word's embedding.
*   A context vector from the attention mechanism.
*   The hidden state of an LSTM.

### [3.5] Data Loader

In [ ]:
# === Transform image based on ImageNet Classification ===
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

# === Data set Innitialization ===
train_ds = FlickrDataset(train_captions_df,dataset_extract_path,tokenizer,transform,max_len=50) # Apply above class to prep in loader
val_ds = FlickrDataset(val_captions_df,dataset_extract_path,tokenizer,transform,max_len=50)
test_ds = FlickrDataset(test_captions_df,dataset_extract_path,tokenizer,transform,max_len=50)


train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,       # Try 2, 4, or 8 (4 is a good start)
    pin_memory=True      # Enables faster host-to-device copies
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
# Build reference lookup (5 references per image) for training
ref_lookup = defaultdict(list)
for _, row in val_captions_df.iterrows():
    ref_lookup[row['image']].append(row['caption'])

### [3.6.1] Hyperparameters

In [ ]:
# === Hyperparameters ===
embed_size  = 512
hidden_size = 512
vocab_size  = tokenizer.vocab_size
feat_dim    = embed_size
num_epochs  = 10
beam_width    = 3
max_len       = 20

# === Set Chechpoint directory  ===
checkpoint_dir = '/content/drive/MyDrive/image_captioning/checkpoints/MobileNetV3'
os.makedirs(checkpoint_dir, exist_ok=True)

###[3.6.2] Load Model, Training + Validation Loop

In [ ]:
# === Models ===
encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderLSTM(embed_size, hidden_size, vocab_size, feat_dim).to(device)

# === Loss & Optimizer ===
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = optim.Adam(
    list(decoder.parameters()) +
    list(encoder.proj.parameters()) +
    list(encoder.bn.parameters()),
    lr=3e-4
)

# === Load latest checkpoint if exists ===
start_epoch = 1
best_metric = -float('inf')  # track BLEU-4

all_ckpts = glob(os.path.join(checkpoint_dir, '*.pth'))
if all_ckpts:
    ckpt_path = sorted(all_ckpts)[-1]  # assumes filenames include epoch or timestamp
    print(f"📂 Loading checkpoint: {ckpt_path}")
    checkpoint = torch.load(ckpt_path, map_location=device)
    encoder.load_state_dict(checkpoint['encoder_state_dict'])
    decoder.load_state_dict(checkpoint['decoder_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_val_loss = checkpoint['val_loss']
    print(f"✅ Restored from epoch {checkpoint['epoch']} (Val Loss: {best_val_loss:.4f})")
else:
    print("ℹ️ No checkpoint found. Training from scratch.")

# === Training + Validation Loop ===
for epoch in range(start_epoch, num_epochs + 1):
    # ——— Training ———
    encoder.train()
    decoder.train()
    train_loss = 0.0

    for images, caps,_ in tqdm(train_loader, desc=f"[Epoch {epoch}] Training"):  # train_loader yields (images, captions)
        images, caps = images.to(device), caps.to(device)
        feats = encoder(images)
        outputs = decoder(feats, caps)  # teacher forcing
        targets = caps[:, 1:]
        loss = criterion(
            outputs.reshape(-1, vocab_size),
            targets.reshape(-1)
        )
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

     # ——— Validation ———
    encoder.eval()
    decoder.eval()
    val_loss = 0.0
    generated_captions = []  # Generated Caption
    reference_captions = []  # list of reference lists

    with torch.no_grad():
        for images, caps, names in tqdm(val_loader, desc=f"[Epoch {epoch}] Validation"):
            images, caps = images.to(device), caps.to(device)

            # 1) Validation loss via teacher forcing
            feats = encoder(images)
            outputs = decoder(feats, caps)
            targets = caps[:, 1:]
            val_loss += criterion(
                outputs.reshape(-1, vocab_size),
                targets.reshape(-1)
            ).item()

            # 2) Beam search generation and collect references
            for i in range(images.size(0)):
                feature = feats[i].unsqueeze(0)  # [1, num_feats, feat_dim]
                # beam search decoding
                sampled_ids = decoder.beam_search(
                    feature,
                    max_len=max_len,
                    start_token_id=tokenizer.cls_token_id,
                    end_token_id=tokenizer.sep_token_id,
                    k=beam_width
                )
                # decode and split into tokens
                cap_str = tokenizer.decode(sampled_ids, skip_special_tokens=True)
                gen_tokens = cap_str.split()
                generated_captions.append(gen_tokens)

                # collect all 5 ground-truth reference captions
                raw_refs = ref_lookup[names[i]]
                refs_tokens = [r.split() for r in raw_refs]
                reference_captions.append(refs_tokens)

    avg_val_loss = val_loss / len(val_loader)

    # ——— BLEU-4 Score ———
    bleu4 = corpus_bleu(
        reference_captions,
        generated_captions,
        weights=(0.25, 0.25, 0.25, 0.25)
    )

    print(
        f"Epoch {epoch:02d} — "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"BLEU-4: {bleu4:.4f}"
    )

    # ——— Checkpointing ———
    if bleu4 > best_metric:
        best_metric = bleu4
        ckpt_path = os.path.join(checkpoint_dir, f'best_epoch_{epoch:02d}.pth')
        torch.save({
            'epoch': epoch,
            'encoder_state_dict': encoder.state_dict(),
            'decoder_state_dict': decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': avg_train_loss,
            'val_loss': avg_val_loss,
            'val_bleu': bleu4,
        }, ckpt_path)
        print(f"  → 🧠 New best model saved (BLEU-4: {bleu4:.4f})")


### [3.6.3] Load Best Model Weights from Checkpoint

In [ ]:
# 1. Re-create the exact same model architectures
embed_size   = 512
hidden_size  = 512
vocab_size   = tokenizer.vocab_size
feat_dim     = embed_size

encoder = EncoderCNN(embed_size).to(device)
decoder = DecoderLSTM(embed_size, hidden_size, vocab_size, feat_dim).to(device)


In [ ]:
# automatically pick the “last” checkpoint in that folder:
checkpoint_dir = '/content/drive/MyDrive/image_captioning/checkpoints/MobileNetV3'
all_ckpts = glob(os.path.join(checkpoint_dir, '*.pth'))
ckpt_path  = sorted(all_ckpts)[-1]

print(f"Loading checkpoint: {ckpt_path}")


In [ ]:
# 2. Load the checkpoint
checkpoint = torch.load(ckpt_path, map_location=device)

In [ ]:
# 3. Restore model weights
encoder.load_state_dict(checkpoint['encoder_state_dict'])
decoder.load_state_dict(checkpoint['decoder_state_dict'])

### [3.7] Model Evaluation

In [ ]:
def build_ref_lookup(captions_df):
    ref_lookup = defaultdict(list)
    for _, row in captions_df.iterrows():
        ref_lookup[row['image']].append(row['caption'])
    return ref_lookup

In [ ]:
val_ref_lookup = build_ref_lookup(val_captions_df)

In [ ]:
# Evaluation on validation set
encoder.eval()
decoder.eval()

generated_captions = []
reference_captions = []
image_names = []
flat_images = []

with torch.no_grad():
    for images, _, names in tqdm(val_loader, desc="Val BLEU Evaluation"):
        images = images.to(device)
        feats = encoder(images)

        for i in range(images.size(0)):
            feature = feats[i].unsqueeze(0)

            # Beam search caption generation
            sampled_ids = decoder.beam_search(
                feature,
                max_len=50,
                start_token_id=tokenizer.cls_token_id,
                end_token_id=tokenizer.sep_token_id,
                k=beam_width
            )
            gen_tokens = tokenizer.decode(sampled_ids, skip_special_tokens=True).split()
            generated_captions.append(gen_tokens)

            # Collect references
            refs = [r.split() for r in val_ref_lookup[names[i]]]
            reference_captions.append(refs)

            # Store image & name for qualitative evaluation
            flat_images.append(images[i].cpu())   # Detach and bring to CPU
            image_names.append(names[i])

### [3.7.1] BLEU Score Validation Set

In [ ]:
# BLEU-1  (unigram precision only)
bleu1 = corpus_bleu(reference_captions, generated_captions, weights=(1.0, 0.0, 0.0, 0.0))

# BLEU-2  (1-gram + 2-gram)
bleu2 = corpus_bleu(reference_captions, generated_captions, weights=(0.5, 0.5, 0.0, 0.0))

# BLEU-3  (1-,2-,3-gram)
bleu3 = corpus_bleu(reference_captions, generated_captions, weights=(1/3, 1/3, 1/3, 0.0))

# BLEU-4  (1- through 4-gram)
bleu4 = corpus_bleu(reference_captions, generated_captions, weights=(0.25, 0.25, 0.25, 0.25))

print(f"Validation BLEU-1: {bleu1:.4f}")
print(f"Validation BLEU-2: {bleu2:.4f}")
print(f"Validation BLEU-3: {bleu3:.4f}")
print(f"Validation BLEU-4: {bleu4:.4f}")

### [3.7.2] BLEU Score Testing Set

In [ ]:
test_ref_lookup = build_ref_lookup(test_captions_df)

In [ ]:
encoder.eval()
decoder.eval()

generated_captions = []
reference_captions = []
image_names = []
flat_images = []

with torch.no_grad():
    for images, _, names in tqdm(test_loader, desc="Test BLEU Evaluation"):
        images = images.to(device)
        feats = encoder(images)

        for i in range(images.size(0)):
            feature = feats[i].unsqueeze(0)

            # Beam search caption generation
            sampled_ids = decoder.beam_search(
                feature,
                max_len=20,
                start_token_id=tokenizer.cls_token_id,
                end_token_id=tokenizer.sep_token_id,
                k=beam_width
            )
            gen_tokens = tokenizer.decode(sampled_ids, skip_special_tokens=True).split()
            generated_captions.append(gen_tokens)

            # Collect references
            refs = [r.split() for r in test_ref_lookup[names[i]]]
            reference_captions.append(refs)

            # Store image & name for qualitative evaluation
            flat_images.append(images[i].cpu())   # Detach and bring to CPU
            image_names.append(names[i])

In [ ]:
# BLEU-1  (unigram precision only)
bleu1 = corpus_bleu(reference_captions, generated_captions, weights=(1.0, 0.0, 0.0, 0.0))

# BLEU-2  (1-gram + 2-gram)
bleu2 = corpus_bleu(reference_captions, generated_captions, weights=(0.5, 0.5, 0.0, 0.0))

# BLEU-3  (1-,2-,3-gram)
bleu3 = corpus_bleu(reference_captions, generated_captions, weights=(1/3, 1/3, 1/3, 0.0))

# BLEU-4  (1- through 4-gram)
bleu4 = corpus_bleu(reference_captions, generated_captions, weights=(0.25, 0.25, 0.25, 0.25))

print(f"Test BLEU-1: {bleu1:.4f}")
print(f"Test BLEU-2: {bleu2:.4f}")
print(f"Test BLEU-3: {bleu3:.4f}")
print(f"Test BLEU-4: {bleu4:.4f}")

BLEU scores tell you how much n-gram overlap there is between your model’s captions and the human references:

+ BLEU-1 (unigrams): 0.6169. The model is capturing relevant unigrams (individual words) quite well.

+ BLEU-2 to BLEU-4: The score steadily drops, as expected, since matching bigrams, trigrams, and 4-grams becomes harder. This usually means: The model captures content words well (objects, actions), but struggles with grammar, syntax, or sequence fluency.


###[3.7.3] Qualitative Assessment

In [ ]:
# === Show Qualitative Samples ===
num_samples = 5
indices = random.sample(range(len(flat_images)), num_samples)

for i in indices:
    img = flat_images[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # Unnormalize
    img = img.clip(0, 1)

    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Generated:\n{' '.join(generated_captions[i])}\n\n" +
        "References:\n" + '\n'.join(['- ' + ' '.join(ref) for ref in reference_captions[i]]),
        fontsize=8,
        wrap=True  # only needed if using interactive display backends
    )
    plt.show()

## 4. Experiment 2: Resnet Encoder + LSTM

**Key Points:**
Previous experiment indicates that the model fail top capture certain context of the image possibly due to using a smaller model (MobileNetV3). In this experiment we will:

+ Use Resnet50 as the Encoder while keep the same Decoder
+ Define a Class ImageCaptioningModel that wraps both encoder and decoder for a cleaner code.
+ Increase max_length of beam search to 50 (to generate longer descripotion)


### [4.1] Define Class definition

In [ ]:
class ResnetEncoder(nn.Module):
    def __init__(self, embed_size):
        super().__init__()
        resnet = models.resnet50(pretrained=True) # Initiate ResNet50 model
        modules = list(resnet.children())[:-2]  # remove avgpool & fc
        self.backbone = nn.Sequential(*modules)
        self.proj = nn.Linear(resnet.fc.in_features, embed_size)
        self.bn = nn.BatchNorm1d(embed_size, momentum=0.01)

    def forward(self, images):
        feat_map = self.backbone(images)             # (B, 2048, H, W)
        B, C, H, W = feat_map.shape
        feat_map = feat_map.permute(0, 2, 3, 1)      # (B, H, W, C)
        feat_map = feat_map.view(B, H*W, C)          # (B, N=H*W, C)
        proj_map = self.proj(feat_map)               # (B, N, embed_size)
        return proj_map

###[4.2] Resnet Encoder

In [ ]:
# === Combine into one model wrapper ===
class ImageCaptioningModel(nn.Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, images, captions):
        feats = self.encoder(images)
        return self.decoder(feats, captions)

    @torch.no_grad()
    def generate(self, images, max_len, start_token_id, end_token_id, k):
        # Beam search wrapper
        feats = self.encoder(images)
        captions = []
        for i in range(images.size(0)):
            cap_ids = self.decoder.beam_search(
                feats[i].unsqueeze(0), max_len, start_token_id, end_token_id, k
            )
            captions.append(cap_ids)
        return captions

### [4.3] Model and Tokenizer

### [4.4] Load Dataset

In [ ]:
image_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

train_ds = FlickrDataset(train_captions_df,dataset_extract_path,tokenizer,image_transform,max_len=50) # Apply above class to prep in loader
val_ds = FlickrDataset(val_captions_df,dataset_extract_path,tokenizer,image_transform,max_len=50)
test_ds = FlickrDataset(test_captions_df,dataset_extract_path,tokenizer,image_transform,max_len=50)


train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=4,       # Try 2, 4, or 8 (4 is a good start)
    pin_memory=True      # Enables faster host-to-device copies
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

test_loader = DataLoader(
    test_ds,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=4,
    pin_memory=True
)

In [ ]:
# Build reference lookup for BLEU computation during training loop
ref_lookup = defaultdict(list)
for _, row in val_captions_df.iterrows():
    ref_lookup[row['image']].append(row['caption'])

### [4.5] Hyperparameters

In [ ]:
# === Hyperparameters ===
embed_size   = 512
hidden_size  = 512
vocab_size   = tokenizer.vocab_size
feat_dim     = embed_size
num_epochs   = 10
beam_width   = 3
max_len      = 50

checkpoint_dir = '/content/drive/MyDrive/image_captioning/checkpoints/Resnet'
os.makedirs(checkpoint_dir, exist_ok=True)

### [4.4] Training Loop

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# === Instantiate model ===
encoder = ResnetEncoder(embed_size).to(device)
decoder = DecoderLSTM(embed_size, hidden_size, vocab_size, feat_dim).to(device)
model = ImageCaptioningModel(encoder, decoder).to(device)


# === Loss & Optimizer ===
pad_idx = tokenizer.pad_token_id
criterion = nn.CrossEntropyLoss(ignore_index=pad_idx)
optimizer = optim.Adam(
    list(model.decoder.parameters()) +
    list(model.encoder.proj.parameters()) +
    list(model.encoder.bn.parameters()),
    lr=3e-4
)

# === Checkpoint loading ===
start_epoch = 1
best_bleu4 = 0.0
all_ckpts = glob(os.path.join(checkpoint_dir, '*.pth'))
if all_ckpts:
    ckpt_path = sorted(all_ckpts)[-1]
    print(f"📂 Loading checkpoint: {ckpt_path}")
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_bleu4 = ckpt['val_bleu']
    print(f"✅ Restored from epoch {ckpt['epoch']} (BLEU-4: {best_bleu4:.4f})")
else:
    print("ℹ️ No checkpoint found. Training from scratch.")

# === Training + Validation Loop ===
for epoch in range(start_epoch, num_epochs + 1):
    # Training
    model.train()
    train_loss = 0.0
    for images, caps, _ in tqdm(train_loader, desc=f"[Epoch {epoch}] Training"):
        images, caps = images.to(device), caps.to(device)
        outputs = model(images, caps)
        targets = caps[:, 1:]
        loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0.0

    generated = []
    references = []

    with torch.no_grad():
        for images, caps, names in tqdm(val_loader, desc=f"[Epoch {epoch}] Validation"):
            images, caps = images.to(device), caps.to(device)
            # Loss
            outputs = model(images, caps)
            targets = caps[:, 1:]
            val_loss += criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1)).item()
            # Generation
            caps_ids = model.generate(
                images,
                max_len=max_len,
                start_token_id=tokenizer.cls_token_id,
                end_token_id=tokenizer.sep_token_id,
                k=beam_width
            )
            for i, ids in enumerate(caps_ids):
                cap_str = tokenizer.decode(ids, skip_special_tokens=True)
                generated.append(cap_str.split())
                raw_refs = ref_lookup[names[i]]
                references.append([r.split() for r in raw_refs])

    avg_val_loss = val_loss / len(val_loader)
    bleu4 = corpus_bleu(references, generated, weights=(0.25,)*4)
    print(f"Epoch {epoch:02d} — Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | BLEU-4: {bleu4:.4f}")

    # Checkpoint
    if bleu4 > best_bleu4:
        best_bleu4 = bleu4
        save_path = os.path.join(checkpoint_dir, f'best_epoch_{epoch:02d}.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_bleu': bleu4
        }, save_path)
        print(f"  → 🧠 New best model saved (BLEU-4: {bleu4:.4f})")

### [4.5] Model Evaluation :

In [ ]:
def build_ref_lookup(captions_df):
    ref_lookup = defaultdict(list)
    for _, row in captions_df.iterrows():
        ref_lookup[row['image']].append(row['caption'])
    return ref_lookup


### [4.5.1] Load Model Weight From Checkpoint

In [ ]:
# 1. Re-create the exact same model architectures
embed_size   = 512
hidden_size  = 512
vocab_size   = tokenizer.vocab_size
feat_dim     = embed_size
num_epochs   = 10
beam_width   = 3
max_len      = 50

checkpoint_dir = '/content/drive/MyDrive/image_captioning/checkpoints/Resnet'
os.makedirs(checkpoint_dir, exist_ok=True)

In [ ]:
# === Setup ===
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
checkpoint_dir = '/content/drive/MyDrive/image_captioning/checkpoints/Resnet'

# Instantiate model
encoder = ResnetEncoder(embed_size).to(device)
decoder = DecoderLSTM(embed_size, hidden_size, vocab_size, feat_dim).to(device)
model = ImageCaptioningModel(encoder, decoder).to(device)

# Optimizer and loss
optimizer = optim.Adam(
    list(model.decoder.parameters()) +
    list(model.encoder.proj.parameters()) +
    list(model.encoder.bn.parameters()),
    lr=3e-4
)

# Load best checkpoint
all_ckpts = glob(os.path.join(checkpoint_dir, '*.pth'))
assert all_ckpts, "❌ No checkpoint found!"

ckpt_path = sorted(all_ckpts)[-1]
print(f"📂 Loading checkpoint: {ckpt_path}")
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['model_state_dict'])
optimizer.load_state_dict(ckpt['optimizer_state_dict'])
print(f"✅ Restored model from epoch {ckpt['epoch']} (BLEU-4: {ckpt['val_bleu']:.4f})")


In [ ]:
val_ref_lookup = build_ref_lookup(val_captions_df)

In [ ]:
# === Evaluate on validation set===
model.eval()
generated = []
references = []
flat_images = []
image_names = []

with torch.no_grad():
    for images, _, names in tqdm(val_loader, desc="Val Evaluation"):
        images = images.to(device)
        feats = model.encoder(images)

        for i in range(images.size(0)):
            feature = feats[i].unsqueeze(0)  # [1, num_feats, feat_dim]
            sampled_ids = model.decoder.beam_search(
                feature,
                max_len=max_len,
                start_token_id=tokenizer.cls_token_id,
                end_token_id=tokenizer.sep_token_id,
                k=beam_width
            )
            cap_str = tokenizer.decode(sampled_ids, skip_special_tokens=True)
            generated.append(cap_str.split())

            raw_refs = val_ref_lookup[names[i]]
            references.append([r.split() for r in raw_refs])

            flat_images.append(images[i].cpu())
            image_names.append(names[i])

bleu4 = corpus_bleu(references, generated, weights=(0.25,) * 4)
print(f"🧾 BLEU-4 Score: {bleu4:.4f}")


### [4.5.2] BLEU Score Validation Set

In [ ]:
from nltk.translate.bleu_score import corpus_bleu

# BLEU-1  (unigram precision only)
bleu1 = corpus_bleu(references, generated, weights=(1.0, 0.0, 0.0, 0.0))

# BLEU-2  (1-gram + 2-gram)
bleu2 = corpus_bleu(references, generated, weights=(0.5, 0.5, 0.0, 0.0))

# BLEU-3  (1-,2-,3-gram)
bleu3 = corpus_bleu(references, generated, weights=(1/3, 1/3, 1/3, 0.0))

# BLEU-4  (1- through 4-gram)
bleu4 = corpus_bleu(references, generated, weights=(0.25, 0.25, 0.25, 0.25))

print(f"Validation BLEU-1: {bleu1:.4f}")
print(f"Validation BLEU-2: {bleu2:.4f}")
print(f"Validation BLEU-3: {bleu3:.4f}")
print(f"Validation BLEU-4: {bleu4:.4f}")


###[4.5.3] BLEU Score Testing Set

In [ ]:
test_ref_lookup = build_ref_lookup(test_captions_df)

In [ ]:
# === Evaluate on Test set===
encoder.eval()
decoder.eval()

generated= []
reference = []
image_names = []
flat_images = []

with torch.no_grad():
    for images, _, names in tqdm(test_loader, desc="Test BLEU Evaluation"):
        images = images.to(device)
        feats = encoder(images)

        for i in range(images.size(0)):
            feature = feats[i].unsqueeze(0)

            # Beam search caption generation
            sampled_ids = decoder.beam_search(
                feature,
                max_len=50,
                start_token_id=tokenizer.cls_token_id,
                end_token_id=tokenizer.sep_token_id,
                k=beam_width
            )
            gen_tokens = tokenizer.decode(sampled_ids, skip_special_tokens=True).split()
            generated.append(gen_tokens)

            # Collect references
            refs = [r.split() for r in test_ref_lookup[names[i]]]
            reference.append(refs)

            # Store image & name for qualitative evaluation
            flat_images.append(images[i].cpu())   # Detach and bring to CPU
            image_names.append(names[i])

In [ ]:
# BLEU-1  (unigram precision only)
bleu1 = corpus_bleu(reference, generated, weights=(1.0, 0.0, 0.0, 0.0))

# BLEU-2  (1-gram + 2-gram)
bleu2 = corpus_bleu(reference, generated, weights=(0.5, 0.5, 0.0, 0.0))

# BLEU-3  (1-,2-,3-gram)
bleu3 = corpus_bleu(reference, generated, weights=(1/3, 1/3, 1/3, 0.0))

# BLEU-4  (1- through 4-gram)
bleu4 = corpus_bleu(reference, generated, weights=(0.25, 0.25, 0.25, 0.25))

print(f"Test BLEU-1: {bleu1:.4f}")
print(f"Test BLEU-2: {bleu2:.4f}")
print(f"Test BLEU-3: {bleu3:.4f}")
print(f"Test BLEU-4: {bleu4:.4f}")

###[4.5.4] Qualitative Evaluation

In [ ]:
# === Show Qualitative Samples ===
num_samples = 5
indices = random.sample(range(len(flat_images)), num_samples)

for i in indices:
    img = flat_images[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # Unnormalize
    img = img.clip(0, 1)

    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Generated:\n{' '.join(generated[i])}\n\n" +
        "References:\n" + '\n'.join(['- ' + ' '.join(ref) for ref in reference[i]]),
        fontsize=8,
        wrap=True  # only needed if using interactive display backends
    )
    plt.show()

## 5. Experiment 3: Blip

### [5.1] Dataset & Early stopping class

In [ ]:
class BlipFlickrDataset(Dataset):
    def __init__(self, df, img_dir, processor, max_len=50):   # Init Max token per caption with 50
        self.df        = df.reset_index(drop=True)            # Init a dataframe with: 'image' and 'caption'
        self.img_dir   = img_dir                              # The directory containing the image files
        self.processor = processor                            # A HuggingFace BLIPProcessor object that tokenizes the text and preprocesses the image
        self.max_len   = max_len                              # The maximum number of tokens for the caption

    def __len__(self):
        return len(self.df)             # Return number of samples (rows) in dataset

    def __getitem__(self, idx):         # Apply Blip processor
        row     = self.df.iloc[idx]
        img_path= os.path.join(self.img_dir, row['image'])
        image   = Image.open(img_path).convert('RGB')
        caption = row['caption']

        # BLIP processor: returns pixel_values, input_ids, attention_mask
        encoding = self.processor(
            images=image,
            text=caption,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=self.max_len
        )
        # remove batch dim
        encoding = {k: v.squeeze(0) for k, v in encoding.items()}

        # store raw caption for reference
        encoding['raw_caption'] = caption
        # optional: store image name for tracking
        encoding['image_name']   = row['image']
        return encoding

def blip_collate_fn(batch):
    keys = batch[0].keys()
    collated = {}
    for k in keys:
        if k in ("raw_caption", "image_name"):
            # keep as list of strings
            collated[k] = [b[k] for b in batch]
        else:
            # tensor keys: stack
            collated[k] = torch.stack([b[k] for b in batch])
    return collated

In [ ]:
class EarlyStopping:
    def __init__(self, patience=3, min_delta=0.0):
        """
        Args:
            patience (int): Number of epochs to wait before stopping.
            min_delta (float): Minimum improvement to reset patience.
        """
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = float('inf')
        self.counter = 0
        self.should_stop = False

    def step(self, current_loss):
        if self.best_loss - current_loss > self.min_delta:
            self.best_loss = current_loss
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True


### [5.2] Blip Model Configuration

In [ ]:
# Model Configurations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_epochs = 10
lr = 5e-5
batch_size = 8
accumulation_steps = 8
checkpoint_dir = "/content/drive/MyDrive/image_captioning/checkpoints"
resume_path = os.path.join(checkpoint_dir, "blip_epoch_02.pt")  # Example
os.makedirs(checkpoint_dir, exist_ok=True)

model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base", use_fast=True)


In [ ]:
model

In [ ]:
processor

In [ ]:
# Load Dataset with class defined from previous cell into Data Loader
train_dataset = BlipFlickrDataset(train_captions_df, dataset_extract_path, processor, max_len=50)
val_dataset   = BlipFlickrDataset(val_captions_df, dataset_extract_path, processor, max_len=50)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=blip_collate_fn, num_workers=4)
val_loader   = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=blip_collate_fn, num_workers=4)

In [ ]:
# Build reference lookup for training
ref_lookup = defaultdict(list)
for _, row in val_captions_df.iterrows():
    ref_lookup[row['image']].append(row['caption'])

### [5.3] Training Loop

In [ ]:
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

In [ ]:
# === SETUP ===
optimizer     = AdamW(model.parameters(), lr=lr)   # An optimizer designed for weight decay (helps generalization).
scaler        = GradScaler()                       # Used for mixed-precision training (with autocast) to save memory and speed up training on GPUs.
early_stopper = EarlyStopping(patience=3)          # Custom class that stops training if validation loss doesn’t improve after 3 epochs
scheduler     = ReduceLROnPlateau(                 # Reduces learning rate when BLEU score has stopped improving
                    optimizer,
                    mode='max',
                    factor=0.5,
                    patience=2,
                    verbose=True
                )

start_epoch = 1
best_bleu   = 0.0

# === RESUME CHECKPOINT IF ANY ===
if os.path.exists(resume_path):
    ckpt = torch.load(resume_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    scaler.load_state_dict(ckpt['scaler_state_dict'])
    best_bleu   = ckpt['best_bleu']
    start_epoch = ckpt['epoch'] + 1
    print(f"🔄 Resumed from epoch {start_epoch}, best BLEU={best_bleu:.2f}")

model.to(device)

# === TRAIN + VAL LOOP ===
for epoch in range(start_epoch, num_epochs + 1):
    # — TRAIN —
    model.train()
    total_train_loss = 0.0

    for step, batch in enumerate(tqdm(train_loader, desc=f"Epoch {epoch:02d} [Train]", leave=False)):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values   = batch["pixel_values"].to(device)
        labels         = input_ids.clone()

        if step % accumulation_steps == 0:
            optimizer.zero_grad()

        with autocast():             # Mixed-precision context to reduce memory and boost speed.
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=labels,
            )
            loss = outputs.loss / accumulation_steps

        scaler.scale(loss).backward()

        if (step + 1) % accumulation_steps == 0:    # Gradients are accumulated for accumulation_steps batches before optimizer.step() is called, allowing effective larger batch size without increasing memory usage.
            scaler.step(optimizer)
            scaler.update()

        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(train_loader)

    # — VALIDATION —
    model.eval()
    total_val_loss = 0.0
    all_preds, all_refs = [], []

    for batch in tqdm(val_loader, desc=f"Epoch {epoch:02d} [Val]", leave=False):
        image_ids      = batch["image_name"]
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        pixel_values   = batch["pixel_values"].to(device)
        labels         = input_ids.clone()

        with torch.inference_mode():
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                pixel_values=pixel_values,
                labels=labels,
            )
            total_val_loss += outputs.loss.item()

            generated = model.generate(
                pixel_values=pixel_values,
                max_length=32,
                num_beams=3,
            )

        preds = tokenizer.batch_decode(generated, skip_special_tokens=True)

        for img_id, pred in zip(image_ids, preds):
            all_preds.append(pred.split())
            all_refs.append([r.split() for r in ref_lookup[img_id]])

    avg_val_loss = total_val_loss / len(val_loader)
    bleu = corpus_bleu(
        all_refs,
        all_preds,
        smoothing_function=SmoothingFunction().method4
    ) * 100

    # — LOG METRICS + SAMPLE —
    print(
        f"Epoch {epoch:02d} → "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val BLEU: {bleu:.2f}"
    )
    print("🔍 Sample Prediction:")
    print(f"   Pred: {' '.join(all_preds[0])}")
    print(f"   Ref : {all_refs[0][0]}")

    # — CHECKPOINT IF IMPROVED —
    if bleu > best_bleu:
        best_bleu = bleu
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'scaler_state_dict': scaler.state_dict(),
            'best_bleu': best_bleu
        }, os.path.join(checkpoint_dir, f"blip_epoch_{epoch:02d}.pt"))
        print(f"💾 Saved best model (BLEU {best_bleu:.2f})")

    scheduler.step(bleu)
    early_stopper.step(avg_val_loss)
    if early_stopper.should_stop:
        print("⏹️ Early stopping triggered!")
        break

    torch.cuda.empty_cache()



*   The model is quite big 2gb saved parameters per EPOCH and take a long time to train
*   Overfitting was persistent, hence to save computational resources it was terminate prematurely.



### [5.4] Model Evaluation

### [5.4.1] Load Model From Checkpoint

In [ ]:
# 2. Device, model & processor
device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model     = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")

# 3. Load checkpoint weights
ckpt = torch.load("/content/drive/MyDrive/image_captioning/checkpoints/blip_epoch_02.pt",
                  map_location=device)

model.load_state_dict(ckpt["model_state_dict"])


In [ ]:
# === Evaluate on validation set===
model.eval()
print("✅ Loaded checkpoint for evaluation.")

generated    = []  # list of list[str]
references   = []  # list of list[list[str]]
image_names  = []  # for later retrieval


with torch.inference_mode():  # faster than no_grad, same memory savings
    for batch in tqdm(val_loader, desc="BLIP Eval"):
        pixel_values = batch["pixel_values"].to(device)
        names        = batch["image_name"]

        # Beam search generation
        outputs = model.generate(
            pixel_values=pixel_values,
            max_length=50,
            num_beams=5,
        )
        preds = processor.batch_decode(outputs, skip_special_tokens=True)

        for i, (name, pred) in enumerate(zip(names, preds)):
            # Tokenized prediction
            generated.append(pred.split())
            image_names.append(name)

            # Multi-reference (tokenized)
            raw_refs = ref_lookup[name]
            references.append([r.split() for r in raw_refs])


### [5.4.2] BLEU Scores on Validation Set

In [ ]:
# 4. Compute BLEU-1 through BLEU-4
bleu1 = corpus_bleu(references, generated, weights=(1.0, 0.0, 0.0, 0.0))
bleu2 = corpus_bleu(references, generated, weights=(0.5, 0.5, 0.0, 0.0))
bleu3 = corpus_bleu(references, generated, weights=(1/3, 1/3, 1/3, 0.0))
bleu4 = corpus_bleu(references, generated, weights=(0.25, 0.25, 0.25, 0.25))

print(f"BLEU-1: {bleu1:.4f}")
print(f"BLEU-2: {bleu2:.4f}")
print(f"BLEU-3: {bleu3:.4f}")
print(f"BLEU-4: {bleu4:.4f}")

###[5.4.3] BLEU Scores on Test Set

In [ ]:
def build_ref_lookup(captions_df):
    ref_lookup = defaultdict(list)
    for _, row in captions_df.iterrows():
        ref_lookup[row['image']].append(row['caption'])
    return ref_lookup

In [ ]:
test_dataset = BlipFlickrDataset(test_captions_df, dataset_extract_path, processor, max_len=50)
test_loader  = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=blip_collate_fn, num_workers=0,pin_memory=False )

test_ref_lookup = build_ref_lookup(test_captions_df)

In [ ]:
import gc

# === Evaluate on Test set ===
model.eval()
print("✅ Loaded checkpoint for evaluation.")

generated    = []  # list of list[str]
references   = []  # list of list[list[str]]
image_names  = []  # for later retrieval
flat_images  = []  # store image tensors (on CPU)

with torch.inference_mode():  # faster than no_grad, same memory savings
    for batch in tqdm(test_loader, desc="BLIP Eval"):
        pixel_values = batch["pixel_values"].to(device)
        names        = batch["image_name"]

        try:
            # Beam search generation
            outputs = model.generate(
                pixel_values=pixel_values,
                max_length=50,
                num_beams=3,
            )
            preds = processor.batch_decode(outputs, skip_special_tokens=True)

            for i, (name, pred) in enumerate(zip(names, preds)):
                # Tokenized prediction
                generated.append(pred.split())
                image_names.append(name)

                # Multi-reference (tokenized)
                raw_refs = test_ref_lookup[name]
                references.append([r.split() for r in raw_refs])

                # Raw image for later visualization
                flat_images.append(batch["pixel_values"][i].cpu())

        except RuntimeError as e:
            print(f"⚠️ Skipping batch due to memory error: {e}")
            continue

        # === Free up memory ===
        del pixel_values, outputs
        torch.cuda.empty_cache()
        gc.collect()


In [ ]:
# 4. Compute BLEU-1 through BLEU-4
bleu1 = corpus_bleu(references, generated, weights=(1.0, 0.0, 0.0, 0.0))
bleu2 = corpus_bleu(references, generated, weights=(0.5, 0.5, 0.0, 0.0))
bleu3 = corpus_bleu(references, generated, weights=(1/3, 1/3, 1/3, 0.0))
bleu4 = corpus_bleu(references, generated, weights=(0.25, 0.25, 0.25, 0.25))

print(f"BLEU-1: {bleu1:.4f}")
print(f"BLEU-2: {bleu2:.4f}")
print(f"BLEU-3: {bleu3:.4f}")
print(f"BLEU-4: {bleu4:.4f}")

### [5.4.2] Qualitative Assessment

In [ ]:
# === Show Qualitative Samples ===
num_samples = 5
indices = random.sample(range(len(flat_images)), num_samples)

for i in indices:
    img = flat_images[i].permute(1, 2, 0).numpy()
    img = img * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # Unnormalize
    img = img.clip(0, 1)

    plt.imshow(img)
    plt.axis("off")
    plt.title(
        f"Generated:\n{' '.join(generated[i])}\n\n" +
        "References:\n" + '\n'.join(['- ' + ' '.join(ref) for ref in references[i]]),
        fontsize=8,
        wrap=True  # only needed if using interactive display backends
    )
    plt.show()